# 🌾 OAN Kenya: 1-Click Cloud GPU Pest Detection Training Engine
### Fine-Tuning Ultralytics YOLOv8 on 12-Class Agricultural Pest Dataset

This Google Colab notebook trains the production **Ultralytics YOLOv8s** architecture across the full 11,502 field images for 50–100 epochs using free **NVIDIA T4 GPU** acceleration in ~25 minutes.

#### 12 Target Agricultural Pest Classes:
`Ants`, `Bees`, `Beetles`, `Caterpillars`, `Earthworms`, `Earwigs`, `Grasshoppers`, `Moths`, `Slugs`, `Snails`, `Wasps`, `Weevils`

### Step 1: Verify NVIDIA GPU Hardware Acceleration

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__} | GPU Active: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

### Step 2: Install Ultralytics & Vision Dependencies

In [ ]:
!pip install -q ultralytics onnx onnxslim

### Step 3: Configure Dataset YAML

In [ ]:
import yaml

dataset_yaml = {
    'path': './dataset', # Point to unzipped dataset folder
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 12,
    'names': [
        'Ants', 'Bees', 'Beetles', 'Caterpillars', 'Earthworms',
        'Earwigs', 'Grasshoppers', 'Moths', 'Slugs', 'Snails',
        'Wasps', 'Weevils'
    ]
}

with open('pest_data_colab.yaml', 'w') as f:
    yaml.dump(dataset_yaml, f)
print("Configured pest_data_colab.yaml")

### Step 4: Execute High-Performance GPU Fine-Tuning

In [ ]:
from ultralytics import YOLO

# Initialize YOLOv8s pre-trained on COCO
model = YOLO('yolov8s.pt')

# Launch training with cosine learning rate & data augmentations
results = model.train(
    data='pest_data_colab.yaml',
    epochs=50,
    imgsz=640,
    batch=32,
    device=0,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    mosaic=0.7,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.4,
    save=True,
    project='runs/train',
    name='oan_kenya_pest_detector'
)

### Step 5: Evaluate on 546 Hold-Out Test Images

In [ ]:
trained_model = YOLO('runs/train/oan_kenya_pest_detector/weights/best.pt')
metrics = trained_model.val(data='pest_data_colab.yaml', split='test', imgsz=640)

print(f"\n🏆 Hold-Out mAP@50: {metrics.box.map50 * 100:.2f}%")
print(f"🏆 Hold-Out mAP@50-95: {metrics.box.map * 100:.2f}%")

### Step 6: Export Edge TFLite & ONNX Models for Kenyan Smallholders

In [ ]:
# Export to ONNX
trained_model.export(format='onnx', imgsz=640)

# Export to TFLite for offline Android smartphone deployment
trained_model.export(format='tflite', imgsz=640)
print("Exported edge mobile models.")

### Step 7: Download Final Trained Weights (best.pt)

In [ ]:
from google.colab import files
files.download('runs/train/oan_kenya_pest_detector/weights/best.pt')